# 第4章　用 `nn.Module` 搭神经网络 & 分类

只用直线学不出弯曲的边界。**堆叠多层、在层间夹入激活函数（非线性）**，就成了神经网络。
本章用 `nn.Module` 搭模型，解决**分类问题**（猜类别）。

本章目标：能定义自己的模型类，并正确使用分类损失 `CrossEntropyLoss`。

> **本笔记使用方法**：从上到下 `Shift + Enter`。多数章节不需要 GPU；较重的章节会说明。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 4-1. 部件：`nn.Linear` / 激活 / `nn.Sequential`

- `nn.Linear(in, out)`：全连接层（上一章直线的多维版）。
- 激活函数 `nn.ReLU()` 等：提供**非线性**。没有它，叠多少层都等于一条直线。
- `nn.Sequential(...)`：把层按顺序排好的简便写法。

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(2, 16),   # 输入2维 -> 16
    nn.ReLU(),          # 非线性
    nn.Linear(16, 2),   # -> 输出2（2个类别的分数）
)
print(model)

x = torch.randn(5, 2)        # 5个样本, 每个2维
print("输出 shape:", model(x).shape)   # (5, 2)

## 4-2. 用类定义 `nn.Module`（实战基本形）

复杂模型用继承 `nn.Module` 的类来写。约定只有两条：
- `__init__` 里**创建要用的层**（别忘了 `super().__init__()`）。
- `forward` 里写**数据的流动**。

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim=2, hidden=16, out_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.act(self.fc1(x))   # 第1层 -> ReLU
        x = self.fc2(x)             # 第2层（输出。这里不加激活）
        return x

model = MLP()
print(model)

## 4-3. 造分类数据（两簇点）
在二维平面上造一簇左下角（类0）和一簇右上角（类1）的点。

In [ ]:
torch.manual_seed(0)
n = 200
c0 = torch.randn(n, 2) + torch.tensor([-2.0, -2.0])   # 类0
c1 = torch.randn(n, 2) + torch.tensor([ 2.0,  2.0])   # 类1
X = torch.cat([c0, c1], dim=0)                        # (400, 2)
yv = torch.cat([torch.zeros(n), torch.ones(n)]).long()  # 标签是整数 (400,)
print("X:", X.shape, " y:", yv.shape, " y的取值:", yv.unique())

## 4-4. 分类损失：`CrossEntropyLoss`（易踩坑）

分类用 `nn.CrossEntropyLoss`。**这里是初学者最大的坑**：

- 模型输出直接给**原始分数（logits）**。**不要自己加 `softmax`**（损失函数内部会做）。
- 真实标签是**类别编号的整数（long）**。不要 one-hot。

> softmax 把分数变成"概率（和为1）"。只有想看预测类别时，最后才用它。

In [ ]:
model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

for epoch in range(100):
    optimizer.zero_grad()
    logits = model(X)            # 原始分数 (400, 2)。不要 softmax！
    loss = criterion(logits, yv) # 真实标签是整数
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}: loss={loss.item():.4f}")

## 4-5. 计算准确率（accuracy）＆ 评估的规矩

评估时：
- `model.eval()`（切到评估模式。Dropout 等行为会变）
- `with torch.no_grad():`（不需要梯度 → 更快、更省内存）
- 预测类别是 logits **最大值的位置** = `argmax(dim=1)`。

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X)
    probs = torch.softmax(logits, dim=1)   # 想要概率就在这里
    pred = logits.argmax(dim=1)            # 预测类别（0 或 1）
    acc = (pred == yv).float().mean()

print("前3个的概率:\n", probs[:3])
print("准确率 accuracy =", acc.item())

## 4-6. 可视化决策边界（附加）
看模型把平面如何分成两色。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

xx, yy = torch.meshgrid(torch.linspace(-6, 6, 200),
                        torch.linspace(-6, 6, 200), indexing="xy")
grid = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
with torch.no_grad():
    zz = model(grid).argmax(dim=1).reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx.numpy(), yy.numpy(), zz.numpy(), alpha=0.3)
plt.scatter(X[:, 0], X[:, 1], c=yv, s=10, cmap="bwr")
plt.title("Decision boundary"); plt.show()

## 练习 4
1. 把类别增加到**3个**（再加一簇点，输出改成 `nn.Linear(16, 3)`，标签为 0/1/2）。`CrossEntropyLoss` 不变即可用。
2. 把隐藏层宽度 `hidden` 改成 4 / 64，比较边界的平滑度。
3. 去掉激活 `ReLU`（`forward` 里直接 `self.fc1(x)`）会发现边界变成直线＝非线性的作用。

In [ ]:
# 在这里写你自己的代码并运行
